[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/delvalle-creator/IA-Cuencas-2026/blob/main/02_PRACTICA/TP3/04_Scripts_Colab/02_superresolucion_S2DR4.ipynb)

> **Para abrir en Colab sin GitHub:** entre a https://colab.research.google.com -> pestana **Subir** (Upload) -> seleccione este archivo `.ipynb`.

---

# TP3 - Superresolucion de Sentinel-2 con S2DR4
## Notebook de Google Colab (metodo real)

**Area de estudio:** sector de Bahia Blanca (escena Sentinel-2 T20HNC)

S2DR4 lleva la imagen Sentinel-2 de **10 m/px a 1 m/px** (12 bandas), con buena reduccion de
nubes y sombras. Solo hay que indicar **coordenadas + fecha**; el modelo descarga la escena.

> ### ORDEN DE EJECUCION (importante)
> 1. Activar GPU (T4).
> 2. Paso 1: instalar el paquete.
> 3. **Reiniciar la sesion** (Paso 1b) -> evita el error de NumPy y mantiene el Drive montado.
> 4. Paso 2: montar Drive.  5. Paso 3: coordenadas.  6. Paso 4: inferencia.
> 7. Paso 5: guardar en Drive (recursivo) y descargar al disco.

## Paso 1 - Activar la GPU e instalar el paquete S2DR4

Active la GPU: menu `Entorno de ejecucion` -> `Cambiar tipo de entorno de ejecucion` -> **GPU (T4)**.
Luego ejecute la celda (instala el paquete; tarda ~1 min).

In [ ]:
# Instalacion del paquete oficial S2DR4 (wheel de gamma.earth)
!pip -q install https://storage.googleapis.com/0x7ff601307fa5/s2dr4-20260126.1-cp312-cp312-linux_x86_64.whl
print('S2DR4 instalado. AHORA reinicie la sesion (ver Paso 1b) antes de continuar.')

## Paso 1b - Reiniciar la sesion (HACERLO UNA VEZ, despues de instalar)

En el menu: `Entorno de ejecucion` -> `Reiniciar sesion`. Confirme con **Si**.

Esto recarga las librerias recien instaladas (evita el error
`cannot import name '_center' from numpy._core.umath`) y deja el entorno limpio.

**Importante:** despues de reiniciar, NO vuelva a ejecutar el Paso 1. Continue desde el Paso 2.

> (Opcional) Si prefiere reiniciar por codigo, descomente y ejecute la linea siguiente; la
> sesion se reinicia sola y luego continua desde el Paso 2.

In [ ]:
# import os; os.kill(os.getpid(), 9)   # (opcional) reinicia la sesion automaticamente

## Paso 2 - Conectar Google Drive

Asi las salidas se guardan en su Drive. Ejecute la celda y, en el cuadro que aparece, haga clic
en **Conectar con Google Drive**, elija su cuenta y acepte todos los permisos.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/TP3_S2DR4'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Drive montado. Las salidas se guardaran en:', DRIVE_OUT)

## Paso 3 - Definir el punto (coordenadas) y la fecha

Indique el centro del area como `(longitud, latitud)` y la fecha exacta de una escena
Sentinel-2. Para el punto con nubes, descomente la linea alternativa.

In [ ]:
# Centro del area de interes: (longitud, latitud)
# --- Punto DESPEJADO (referencia) ---
# lonlat = (-62.0653, -38.3150)    # zona despejada (escena T20HNC, 30-abr-2026)
# --- Pin 1: SOMBRA DE NUBES -> ACTIVO (correr este primero) ---
lonlat = (-62.18729, -38.21575)    # Pin 1: sombra de nubes
# --- Pin 2: NUBES DISPERSAS (despues: quite el # de esta linea y comente el Pin 1) ---
# lonlat = (-62.36289, -38.60399)  # Pin 2: nubes dispersas
date   = '2026-04-30'              # escena Sentinel-2 con ~38% de nubes
print('Procesando', lonlat, 'fecha', date)

## Paso 4 - Ejecutar la superresolucion

Descarga la escena, descarga el modelo (~841 MB la primera vez) y superresuelve a 1 m.
En GPU T4 tarda unos minutos. Las salidas quedan en `/content/output/`.

In [ ]:
import s2dr4.inferutils
s2dr4.inferutils.test(lonlat, date)
print('Listo. Salidas en /content/output/')

## Paso 5 - Guardar en Drive (recursivo) y descargar al disco

S2DR4 guarda los archivos en subcarpetas (`/content/output/AR/T20HNC/...`). Por eso la copia a
Drive debe ser **recursiva**. Esta celda copia todo el arbol y **lista** los .tif copiados para
verificar que realmente quedaron en Drive.

In [ ]:
import shutil, glob, os
# Copia RECURSIVA de todo el arbol de salidas a Drive
destino = os.path.join(DRIVE_OUT, 'output')
shutil.copytree('/content/output', destino, dirs_exist_ok=True)

# Verificacion: listar los GeoTIFF que quedaron en Drive
tifs_drive = sorted(glob.glob(os.path.join(destino, '**', '*.tif'), recursive=True))
print('GeoTIFF copiados a Drive:', len(tifs_drive))
for t in tifs_drive:
    print('  ', t)
print('\nUbicacion en Drive:', destino)

### (Opcional) Descargar una copia local (.zip) al disco
Genera un ZIP con todas las salidas y lo descarga a la carpeta Descargas del navegador.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/s2dr4_resultados', 'zip', '/content/output')
print('ZIP listo, descargando...')
files.download('/content/s2dr4_resultados.zip')

## Paso 6 - Areas grandes (mosaico)

Cada corrida cubre un cuadrante limitado. Para cubrir mas terreno, se procesan varios centros
y luego se arma el mosaico en QGIS (`Raster -> Miscelanea -> Combinar`) o con gdal_merge.

In [ ]:
# Ejemplo: recorrer una grilla de centros (ajustela a su area)
centros = [
    (-62.0853, -38.3250), (-62.0653, -38.3250),
    (-62.0853, -38.3050), (-62.0653, -38.3050),
]
date = '2026-04-30'
import s2dr4.inferutils
for ctr in centros:
    print('Procesando centro', ctr)
    s2dr4.inferutils.test(ctr, date)
print('Listo. Combine los GeoTIFF de /content/output/ en un mosaico (QGIS o gdal_merge).')

## Paso 7 - Comparacion en QGIS e informe

1. Descargue de Drive (o del disco) los productos de 1 m.
2. En **QGIS**, cargue el producto de 1 m y una referencia de 10 m, y use el complemento de
   cortina (**@map-swipe-tool**) sobre el cauce del arroyo, zonas urbanas y parcelas.
3. Documente en el informe: comparacion 10 m vs 1 m, manejo de nubes/sombras, y limitaciones
   (detalle estimado por IA; validar con ortofotos).